# 01 — Tor setup and safe fetch

**Goal.** Get Tor running locally, route HTTP through its SOCKS proxy, fetch a single onion page, and rotate circuits between requests. Everything downstream depends on this working.

**Scope.** Single page, single circuit, single thread. No crawling logic yet — that's notebook 02.

**Prereqs.**
```bash
sudo apt install tor
pip install stem requests[socks] pysocks
```

## Ethics & legal checklist (read before running)

- Dark-web collection is **jurisdiction-dependent**. Confirm what you are allowed to fetch where you operate.
- Do not fetch CSAM-adjacent content. The seed list in notebook 02 filters Ahmia categories explicitly; do not bypass it.
- Do not authenticate to forums or marketplaces from this notebook. Public, unauthenticated pages only.
- Run inside a dedicated VM. Do not use your daily-driver browser profile or system Tor instance for unrelated traffic.
- Respect rate limits. The fetch helper sleeps between calls — leave it in.


## 1. Confirm Tor is running

Tor's default SOCKS port is 9050; its control port (which we use to rotate circuits) is 9051 but is **not** enabled by default. We will configure it below.

In [1]:
import subprocess

# Is the tor service running?
out = subprocess.run(["systemctl", "is-active", "tor"], capture_output=True, text=True)
print("tor service:", out.stdout.strip() or out.stderr.strip())

tor service: active


If the service is inactive:

```bash
sudo systemctl start tor
```

## 2. Enable the control port

We need the control port to ask Tor for a fresh circuit. Add these two lines to `/etc/tor/torrc` once and restart the service:

```
ControlPort 9051
CookieAuthentication 1
```

Then `sudo systemctl restart tor`. The cookie file is owned by `debian-tor`; either add your user to that group or run the control commands as root in a dedicated VM.

## 3. Fetch a clearnet page through Tor

We start with `https://check.torproject.org/` because it tells us whether the request actually went through Tor. If this fails, nothing else in this repo will work.

In [2]:
import requests

TOR_PROXIES = {
    "http":  "socks5h://127.0.0.1:9050",
    "https": "socks5h://127.0.0.1:9050",
}

r = requests.get("https://check.torproject.org/", proxies=TOR_PROXIES, timeout=30)
print("status:", r.status_code)
print("using tor:", "Congratulations" in r.text)

status: 200
using tor: True


Note the `socks5h://` scheme — the `h` tells the SOCKS client to do DNS resolution **through Tor**. Without it, your DNS leaks to your local resolver, which defeats the purpose.

## 4. Fetch an onion page

We use the **Tor Project's own onion** as the test target — it's the most reliable onion available and the canonical "is my Tor working" destination. The DuckDuckGo onion has been flaky for a while; we used to recommend it but switched.

Note the **120s timeout**: the very first onion connection after Tor starts often takes 60–90s to build the rendezvous circuit. 60s is too tight for a cold start.

If this fails, run the troubleshooting cell below it.

In [3]:
TPO_ONION = "http://2gzyxa5ihm7nsggfxnu52rck2vv4rvmdlkiu3zzui5du4xyclen53wid.onion/"      
r = requests.get(TPO_ONION, proxies=TOR_PROXIES, timeout=120)
print("status:", r.status_code, "| bytes:", len(r.content)) 

status: 200 | bytes: 23012


### Troubleshooting: onion timed out

If you see `ConnectTimeout` / `Connection ... timed out`, work through this in order:

1. **Confirm clearnet-through-Tor still works** — re-run the `check.torproject.org` cell above. If `using tor: True`, Tor itself is healthy and the issue is just this onion or a stale circuit.
2. **The onion is gone or down.** Onions go offline frequently — sometimes for hours, sometimes permanently. The DDG onion (`duckduckgogg42x...onion`) has been unreliable for a while; the Tor Project onion above is your stable check.
3. **Force a fresh circuit and retry** with the cell below. Circuits go stale; new ones often work where old ones didn't.
4. **Don't lower the timeout.** Cold-start onion latency is genuinely 60–90s.

In [5]:
# Troubleshooting: force a new circuit and retry the test onion
# from stem import Signal
# from stem.control import Controller
# import time

# with Controller.from_port(port=9051) as c:
#     c.authenticate()
#     c.signal(Signal.NEWNYM)
# time.sleep(15)  # let the new circuit build

# r = requests.get(TPO_ONION, proxies=TOR_PROXIES, timeout=120)
# print("status:", r.status_code, "| bytes:", len(r.content))

Onion fetches are **slow and unreliable** — connections build a 6-hop circuit and hidden services often go offline. Treat any individual fetch as best-effort and always wrap in retries upstream.

## 5. Rotate circuits with stem

If you keep using the same circuit, the exit node fingerprints you across requests. Rotating after every N fetches (or on failure) reduces this and helps when a particular circuit is dead.

In [7]:
# from stem import Signal
# from stem.control import Controller

# def new_circuit():
#     with Controller.from_port(port=9051) as c:
#         c.authenticate()  # uses the cookie file
#         c.signal(Signal.NEWNYM)

# new_circuit()
# print("requested new circuit")

Tor rate-limits NEWNYM to once every 10 seconds by default. Don't call this in a tight loop.

## 6. The `safe_fetch` helper

Notebooks 02 and 06 import this. It encapsulates: timeout, retry, circuit rotation on failure, and a polite delay.

In [9]:
import time
import random

USER_AGENT = "Mozilla/5.0 (cti_401 research crawler; contact: you@example.com)"

def safe_fetch(url, *, max_retries=3, timeout=60, min_delay=2.0, max_delay=5.0):
    """Fetch `url` through Tor with retries and circuit rotation on failure.
    Returns (status_code, text) or (None, None) if all retries fail.
    Sleeps a random [min_delay, max_delay] before returning so callers can
    just call it in a loop without thinking about pacing.
    """
    headers = {"User-Agent": USER_AGENT}
    for attempt in range(max_retries):
        try:
            r = requests.get(url, proxies=TOR_PROXIES, headers=headers, timeout=timeout)
            time.sleep(random.uniform(min_delay, max_delay))
            return r.status_code, r.text
        except Exception as e:
            print(f"  attempt {attempt+1} failed: {type(e).__name__}: {e}")
            try:
                new_circuit()
                time.sleep(11)  # respect NEWNYM rate limit
            except Exception as ce:
                print(f"  circuit rotation failed: {ce}")
    return None, None

status, text = safe_fetch(TPO_ONION)
print("status:", status, "| chars:", len(text) if text else 0)

status: 200 | chars: 23012


## What's next

Notebook 02 reuses `safe_fetch` to crawl ~100 pages from an Ahmia-derived seed list. Keep this notebook's kernel running, or copy the helper into 02 — it's intentionally small.